# 05. Personalized Mobility Profile & Baseline Modeling

**Project**: AI-Driven Predictive Geofencing and Trajectory Analysis for Proactive Safety Monitoring  
**Stage**: Chunk 5 — Personalized Longitudinal Mobility Profile, Spatial Anchors & Kinematic Baselines  

This notebook demonstrates and inspects the personalized longitudinal mobility profiling system developed in [`ml/src/profile.py`](../src/profile.py) across all 182 GeoLife users.

> **Core Methodological Guarantees**:
> - **No Fabricated Data**: All profile values, spatial anchors, and baselines are learned strictly from historical observations.
> - **Density-Based Spatial Anchors**: Haversine DBSCAN on trajectory endpoints with robust geofence radii (P95 distance floor at 25m) and status tracking (`CONFIRMED` vs `PENDING_CAREGIVER_REVIEW`).
> - **Mathematical Baselines**: Gamma fitting ($N < 30$) where mathematically valid ($x > 0$, $\sigma^2 > 0$); empirical quantiles ($N \ge 30$), P95/P99, and robust scale ($1.4826 \times \text{MAD}$). Explicit handling of $\text{MAD}=0$.
> - **Strict Cold-Start Formula**: $\Delta CV_N = \frac{|CV_N - CV_{N-1}|}{CV_{N-1}}$, transitioning to `ML_DRIVEN` when $\text{trip\_count} \ge 7$ AND $\Delta CV < 0.05$ for 3 consecutive trips.
> - **Zero Future Leakage**: Strictly chronological profile generation at time $T$.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Resolve paths
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'ml' / 'notebooks'
PROCESSED_DIR = NOTEBOOK_DIR.parent / 'data' / 'processed'
PROFILES_PARQUET = PROCESSED_DIR / 'mobility_profiles.parquet'
REPORT_FILE = PROCESSED_DIR / 'mobility_profile_report.json'

print(f'Profiles Parquet: {PROFILES_PARQUET} (exists: {PROFILES_PARQUET.exists()})')
print(f'Report File:     {REPORT_FILE} (exists: {REPORT_FILE.exists()})')

## 1. Load Mobility Profile Audit Report
Inspect population statistics and centralized configuration parameters.

In [ ]:
with open(REPORT_FILE, 'r', encoding='utf-8') as f:
    report = json.load(f)

print(f"Status:                    {report['status']}")
print(f"Total Users Profiled:      {report['dataset_summary']['users_profiled']}")
print(f"ML_DRIVEN Users:           {report['dataset_summary']['ml_driven_users']} ({report['dataset_summary']['ml_driven_percentage']}%)")
print(f"RULE_BASED_FALLBACK Users: {report['dataset_summary']['rule_based_fallback_users']}")
print(f"Total Anchors Discovered:  {report['spatial_anchors']['total_anchors_discovered']}")
print(f"Users with Return Trips:   {report['mobility_trip_statistics']['users_with_return_trips']}")

## 2. Load Profiles Parquet Dataset
Inspect the schema and distribution of cold-start states across users.

In [ ]:
profiles_df = pq.read_table(PROFILES_PARQUET).to_pandas()
print(f"Loaded {len(profiles_df)} profiles from {PROFILES_PARQUET.name}.")
print(profiles_df[['user_id', 'cold_start_status', 'trip_count', 'delta_cv', 'consecutive_stable_trips']].head(10))

## 3. Spatial Anchors Analysis
Inspect anchor count distributions, observation counts, and robust radius derivations.

In [ ]:
# Parse anchor clusters from all users
all_anchors = []
for _, row in profiles_df.iterrows():
    clusters = json.loads(row['anchor_clusters'])
    all_anchors.extend(clusters)

anchors_df = pd.DataFrame(all_anchors)
print(f"Total Spatial Anchors: {len(anchors_df)}")
print("\nAnchor Status Breakdown:")
print(anchors_df['status'].value_counts())

print("\nAnchor Radius (m) Distribution:")
print(anchors_df['radius_m'].describe())

print("\nObservation Count per Anchor Distribution:")
print(anchors_df['observation_count'].describe())

## 4. Personal Mobility & Trip Statistics
Inspect return trip distances, CVs, and maximum historical movement radii.

In [ ]:
trip_stats_list = [json.loads(row['trip_stats']) for _, row in profiles_df.iterrows()]
ts_df = pd.DataFrame(trip_stats_list)
ts_df['user_id'] = profiles_df['user_id']

print("Summary of Trip Statistics Across Users:")
print(ts_df[['avg_return_trip_distance_m', 'cv_return_trip_distance', 'avg_step_speed_mps', 'max_historical_radius_m']].describe())

## 5. Personalized Kinematic Baselines
Inspect distribution types (Gamma vs. Empirical) across validated kinematic features.

In [ ]:
dist_types = {'gamma': 0, 'empirical': 0, 'undefined': 0}
for _, row in profiles_df.iterrows():
    b_dict = json.loads(row['baseline_distribution'])
    for feat, info in b_dict.items():
        dtype = info.get('distribution_type', 'undefined')
        dist_types[dtype] = dist_types.get(dtype, 0) + 1

print("Baseline Distribution Type Tallies Across All Features & Users:")
for dt, cnt in dist_types.items():
    print(f"  {dt:12s}: {cnt:6d}")

## 6. Cold-Start Stability & Stability History
Inspect the trip-by-trip stability convergence curve for a mature user.

In [ ]:
# Inspect stability history for user 000
u0_row = profiles_df[profiles_df['user_id'] == '000'].iloc[0]
u0_hist = json.loads(u0_row['stability_history'])
hist_df = pd.DataFrame(u0_hist)

print(f"User 000 Cold Start Status: {u0_row['cold_start_status']}")
print(f"Total Trips: {len(hist_df)}")
print("\nFirst 10 Trips Stability History:")
print(hist_df[['trip_index', 'path_distance_m', 'cv', 'delta_cv', 'consecutive_stable_trips', 'status']].head(10))

## 7. Sparse vs. Mature Profile Inspection
Compare a profile with few observations vs. a longitudinal mature profile.

In [ ]:
print('=== SPARSE PROFILE EXAMPLE ===')
print(json.dumps(report['profile_examples']['sparse_profile'], indent=2))

print('\n=== MATURE PROFILE EXAMPLE ===')
print(json.dumps(report['profile_examples']['mature_profile'], indent=2))